# Hunyuan3D-2 Paint on Kaggle

**Once:** kaggle.com → your profile → Settings → **Phone verification** (without it Kaggle gives no GPU and no internet).

**Each model:**
1. Kaggle → **Datasets → New Dataset** → drop `s2f.zip` (from the `hunyuan_paint` folder on the Desktop) → name it e.g. `s2f` → Create.
2. Open this notebook (Kaggle → **Code → New Notebook → File → Import Notebook** → this `.ipynb`).
3. Right panel: **Accelerator → GPU T4 x2**, **Internet → On**, **Add Input** → your `s2f` dataset.
4. **Run All**. When it says `painted in ... s`, the file is in the right panel → **Output** → `/kaggle/working/painted.glb` → download.

In [ ]:
# 1. install (~6-8 min)
%cd /kaggle/working
![ -d Hunyuan3D-2 ] || git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /kaggle/working/Hunyuan3D-2
!pip -q install -r requirements.txt 2>&1 | grep -v "dependency resolver\|incompatible" || true
!pip -q install -e . --no-deps
!cd hy3dgen/texgen/custom_rasterizer && pip -q install . --no-build-isolation && echo custom_rasterizer OK
!cd hy3dgen/texgen/differentiable_renderer && pip -q install . --no-build-isolation && echo differentiable_renderer OK

In [ ]:
# 2. find the job (mesh.glb + front/left/back.png) among the attached datasets; zips are unpacked by Kaggle
import glob, os, zipfile
hits = glob.glob('/kaggle/input/**/mesh.glb', recursive=True)
if not hits:  # dataset kept as a zip
    for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall('/kaggle/working/job')
    hits = glob.glob('/kaggle/working/job/**/mesh.glb', recursive=True)
assert hits, 'no mesh.glb found: add the dataset with s2f.zip via Add Input'
JOB = os.path.dirname(hits[0])
print('job:', JOB, sorted(os.listdir(JOB)))

In [ ]:
# 3. paint (~5-10 min); the result is /kaggle/working/painted.glb (Output panel)
import sys, time, json
sys.path.insert(0, '/kaggle/working/Hunyuan3D-2')
import torch, trimesh, diffusers
from PIL import Image
from safetensors.torch import load_file
print('GPU:', torch.cuda.get_device_name(0))

# newer diffusers refuses the painter's own pipeline code unless it is allowed explicitly
_orig = diffusers.DiffusionPipeline.from_pretrained.__func__
def _trusted(cls, *a, **k):
    k['trust_remote_code'] = True
    return _orig(cls, *a, **k)
diffusers.DiffusionPipeline.from_pretrained = classmethod(_trusted)

import hy3dgen.texgen.pipelines as P
from hy3dgen.texgen import Hunyuan3DPaintPipeline
P.mesh_uv_wrap = lambda m: m  # unwrapped on the Mac already

t = time.time()
mesh = trimesh.load(JOB + '/mesh.glb', force='mesh')
print('mesh', len(mesh.faces), 'faces')
imgs = [Image.open(f'{JOB}/{n}.png') for n in ['front', 'left', 'back'] if os.path.isfile(f'{JOB}/{n}.png')]
print('photos', len(imgs))
paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2', subfolder='hunyuan3d-paint-v2-0-turbo')
print('models loaded, painting, %.0f s' % (time.time() - t))
paint(mesh, image=imgs).export('/kaggle/working/painted.glb')
print('painted in %.0f s -> /kaggle/working/painted.glb' % (time.time() - t))